# Tree-of-Thought 24点游戏 (v2)

明确的三轮流程，每轮都是：**Think → Evaluate → Screen**

| 轮次 | 输入 | 输出 | 说明 |
|------|------|------|------|
| 第1轮 | 4个数 | 3个数 | 选两个数运算，保留最优方案 |
| 第2轮 | 3个数 | 2个数 | 再选两个数运算，保留最优方案 |
| 第3轮 | 2个数 | 1个数 | 最后两个数运算，检查是否=24 |

In [8]:
import os, re
from itertools import combinations
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

def qwen_call(prompt, model="qwen-plus"):
    res = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    return res.choices[0].message.content

print("API 初始化完成")

API 初始化完成


## Think：用代码生成所有可能的运算组合

不依赖模型提议，而是用代码穷举，避免模型遗漏或格式错误。

In [ ]:
def think(numbers):
    """
    Think：穷举所有两两运算，返回 [(描述, 剩余数字列表), ...]
    4个数 → 生成所有可能的3个数组合
    3个数 → 生成所有可能的2个数组合
    2个数 → 生成所有可能的1个数结果
    """
    results = []
    ops = [
        ('+', lambda a, b: a + b),
        ('-', lambda a, b: a - b),
        ('*', lambda a, b: a * b),
        ('/', lambda a, b: a / b if b != 0 else None),
    ]

    for i in range(len(numbers)):
        for j in range(len(numbers)):
            if i == j:
                continue
            a, b = numbers[i], numbers[j]
            remaining = [numbers[k] for k in range(len(numbers)) if k != i and k != j]

            for op_name, op_func in ops:
                result = op_func(a, b)
                if result is None:
                    continue

                desc = f"{a} {op_name} {b} = {result}"
                new_numbers = [result] + remaining
                results.append((desc, new_numbers))

    return results

test = think([4, 5, 6, 10])
print(f"4个数的可能运算数: {len(test)}")
print("前5个示例:")
# for desc, nums in test:
for desc, nums in test[:5]:
    print(f"  {desc} → 剩余: {nums}")

4个数的可能运算数: 48
前5个示例:
  4 + 5 = 9 → 剩余: [9, 6, 10]
  4 - 5 = -1 → 剩余: [-1, 6, 10]
  4 * 5 = 20 → 剩余: [20, 6, 10]
  4 / 5 = 0.8 → 剩余: [0.8, 6, 10]
  4 + 6 = 10 → 剩余: [10, 5, 10]
  4 - 6 = -2 → 剩余: [-2, 5, 10]
  4 * 6 = 24 → 剩余: [24, 5, 10]
  4 / 6 = 0.6666666666666666 → 剩余: [0.6666666666666666, 5, 10]
  4 + 10 = 14 → 剩余: [14, 5, 6]
  4 - 10 = -6 → 剩余: [-6, 5, 6]
  4 * 10 = 40 → 剩余: [40, 5, 6]
  4 / 10 = 0.4 → 剩余: [0.4, 5, 6]
  5 + 4 = 9 → 剩余: [9, 6, 10]
  5 - 4 = 1 → 剩余: [1, 6, 10]
  5 * 4 = 20 → 剩余: [20, 6, 10]
  5 / 4 = 1.25 → 剩余: [1.25, 6, 10]
  5 + 6 = 11 → 剩余: [11, 4, 10]
  5 - 6 = -1 → 剩余: [-1, 4, 10]
  5 * 6 = 30 → 剩余: [30, 4, 10]
  5 / 6 = 0.8333333333333334 → 剩余: [0.8333333333333334, 4, 10]
  5 + 10 = 15 → 剩余: [15, 4, 6]
  5 - 10 = -5 → 剩余: [-5, 4, 6]
  5 * 10 = 50 → 剩余: [50, 4, 6]
  5 / 10 = 0.5 → 剩余: [0.5, 4, 6]
  6 + 4 = 10 → 剩余: [10, 5, 10]
  6 - 4 = 2 → 剩余: [2, 5, 10]
  6 * 4 = 24 → 剩余: [24, 5, 10]
  6 / 4 = 1.5 → 剩余: [1.5, 5, 10]
  6 + 5 = 11 → 剩余: [11, 4, 10]
  6 - 5 = 1 

## Evaluate：用模型评估剩余数字凑24的可能性

In [10]:
value_prompt = """评估给定的数字是否可以通过加减乘除运算达到24。
只回答一个词：sure / likely / impossible

10 14
10 + 14 = 24
sure
4 4 10
(10 - 4) * 4 = 24
sure
4 9 11
9 + 11 + 4 = 24
sure
5 7 8
5 + 7 + 8 = 20, (8-5)*7 = 21
likely
11 12
11 + 12 = 23, 12 - 11 = 1
impossible
10 10 10
10 + 10 + 10 = 30, 无法达到24
impossible

{input}
"""

SCORE_MAP = {"sure": 20, "likely": 1, "impossible": 0.001}

def evaluate(numbers):
    """Evaluate：让模型评估这组数字凑24的可能性"""
    numbers_str = ' '.join(str(n) for n in numbers)
    prompt = value_prompt.format(input=numbers_str)
    response = qwen_call(prompt).strip().lower()

    last_line = response.strip().split('\n')[-1].strip()
    for key in SCORE_MAP:
        if key in last_line:
            return SCORE_MAP[key]
    return 0.001

print("Evaluate 函数定义完成")

Evaluate 函数定义完成


## Screen：保留评分最高的方案

In [11]:
def screen(candidates, top_k=5):
    """Screen：按评分排序，保留前 top_k 个方案"""
    candidates.sort(key=lambda x: x[2], reverse=True)
    selected = candidates[:top_k]

    print(f"  筛选后保留 {len(selected)} 个方案:")
    for desc, nums, score in selected:
        nums_str = ' '.join(f'{n:g}' for n in nums)
        print(f"    {desc} → [{nums_str}] (评分: {score})")

    return selected

print("Screen 函数定义完成")

Screen 函数定义完成


## 三轮求解：Think → Evaluate → Screen × 3

In [12]:
def nums_key(numbers):
    """将数字列表转为排序后的元组，用于去重"""
    return tuple(sorted(round(n, 6) for n in numbers))

def solve_game24(numbers, top_k=5):
    """
    三轮求解24点：
    第1轮：4个数 → Think → Evaluate → Screen → 保留top_k组3个数
    第2轮：3个数 → Think → Evaluate → Screen → 保留top_k组2个数
    第3轮：2个数 → Think → 直接检查是否=24（无需API）
    """
    print(f"输入数字: {numbers}\n")

    current_states = [(None, numbers)]

    for round_num in range(1, 4):
        round_name = ["第1轮 (4→3)", "第2轮 (3→2)", "第3轮 (2→1)"][round_num - 1]
        print(f"{'='*50}")
        print(f"{round_name}: Think → Evaluate → Screen")
        print(f"{'='*50}")

        # Think: 穷举所有运算
        all_proposals = []
        for prev_desc, nums in current_states:
            proposals = think(nums)
            for desc, new_nums in proposals:
                full_desc = f"{prev_desc} → {desc}" if prev_desc else desc
                all_proposals.append((full_desc, new_nums))

        print(f"\n  Think: {len(all_proposals)} 种运算方式")

        # 第3轮：直接用数学判断，不需要调API
        if round_num == 3:
            for desc, nums in all_proposals:
                if len(nums) == 1 and abs(nums[0] - 24) < 0.001:
                    print(f"\n{'='*50}")
                    print(f"找到答案！")
                    print(f"{'='*50}")
                    steps = desc.split(' → ')
                    for i, step in enumerate(steps):
                        print(f"  第{i+1}步: {step}")
                    return desc
            print("\n未找到等于24的结果")
            return None

        # Evaluate: 先去重，减少API调用次数
        score_cache = {}
        unique_keys = set()
        for desc, new_nums in all_proposals:
            unique_keys.add(nums_key(new_nums))

        print(f"  Evaluate: {len(unique_keys)} 组不同的数字需要评估（去重前 {len(all_proposals)} 个）...")

        for key in unique_keys:
            score_cache[key] = evaluate(list(key))
            print(f"    {list(key)} → {score_cache[key]}")

        all_candidates = []
        for desc, new_nums in all_proposals:
            score = score_cache[nums_key(new_nums)]
            all_candidates.append((desc, new_nums, score))

        # Screen: 保留最优
        print(f"\n  Screen:")
        selected = screen(all_candidates, top_k)
        current_states = [(desc, nums) for desc, nums, score in selected]
        print()

    return None

## 运行

In [13]:
result = solve_game24([4, 5, 6, 10])

输入数字: [4, 5, 6, 10]

第1轮 (4→3): Think → Evaluate → Screen

  Think: 48 种运算方式
  Evaluate: 36 组不同的数字需要评估（去重前 48 个）...


    [0.5, 4, 6] → 20
    [1.666667, 4, 5] → 1
    [5, 10, 24] → 20
    [1.25, 6, 10] → 1
    [5, 6, 14] → 20
    [4, 10, 11] → 1
    [4, 6, 50] → 0.001
    [2, 5, 10] → 20
    [4, 5, 6] → 20
    [-5, 4, 6] → 20
    [1.5, 5, 10] → 1
    [0.666667, 5, 10] → 1
    [-2, 5, 10] → 20
    [-4, 4, 5] → 20
    [0.833333, 4, 10] → 1
    [4, 5, 60] → 20
    [0.4, 5, 6] → 1
    [4, 6, 15] → 20
    [5, 6, 40] → 1
    [1.2, 4, 10] → 1
    [1, 4, 10] → 1
    [5, 10, 10] → 1
    [2.5, 5, 6] → 20
    [-1, 6, 10] → 20
    [5, 6, 6] → 20
    [6, 9, 10] → 1
    [1, 6, 10] → 20
    [0.6, 4, 5] → 1
    [4, 4, 5] → 20
    [-1, 4, 10] → 20
    [-6, 5, 6] → 20
    [4, 10, 30] → 1
    [4, 5, 16] → 20
    [0.8, 6, 10] → 1
    [6, 10, 20] → 1
    [2.0, 4, 6] → 20

  Screen:
  筛选后保留 5 个方案:
    4 - 5 = -1 → [-1 6 10] (评分: 20)
    4 - 6 = -2 → [-2 5 10] (评分: 20)
    4 * 6 = 24 → [24 5 10] (评分: 20)
    4 + 10 = 14 → [14 5 6] (评分: 20)
    4 - 10 = -6 → [-6 5 6] (评分: 20)

第2轮 (3→2): Think → Evaluate → Screen

  Think: 

In [14]:
# 试试其他数字
result = solve_game24([1, 2, 3, 4])

输入数字: [1, 2, 3, 4]

第1轮 (4→3): Think → Evaluate → Screen

  Think: 48 种运算方式
  Evaluate: 30 组不同的数字需要评估（去重前 48 个）...
    [0.666667, 1, 4] → 1
    [0.333333, 2, 4] → 1
    [-1, 1, 4] → 20
    [-1, 3, 4] → 20
    [0.5, 3, 4] → 20
    [1, 3, 6] → 20
    [1, 4, 5] → 1
    [2, 3, 3] → 20
    [0.5, 1, 3] → 20
    [-3, 2, 3] → 20
    [-2, 1, 3] → 20
    [2, 2, 4] → 20
    [0.25, 2, 3] → 20
    [1, 2, 7] → 1
    [1, 3, 8] → 20
    [3, 3, 4] → 20
    [-2, 2, 4] → 20
    [2, 3, 5] → 20
    [1, 1, 2] → 0.001
    [2, 4, 4] → 20
    [0.75, 1, 2] → 20
    [1, 1.5, 4] → 1
    [-1, 1, 2] → 20
    [1, 2, 3] → 20
    [1, 4, 6] → 20
    [1, 3, 4] → 20
    [1, 2, 12] → 20
    [1, 1.333333, 2] → 1
    [2, 3, 4] → 20
    [1, 1, 4] → 0.001

  Screen:
  筛选后保留 5 个方案:
    1 + 2 = 3 → [3 3 4] (评分: 20)
    1 - 2 = -1 → [-1 3 4] (评分: 20)
    1 * 2 = 2 → [2 3 4] (评分: 20)
    1 / 2 = 0.5 → [0.5 3 4] (评分: 20)
    1 + 3 = 4 → [4 2 4] (评分: 20)

第2轮 (3→2): Think → Evaluate → Screen

  Think: 120 种运算方式
  Evaluate: 61 组不同的数